In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D11 — Siyaram Silk Mills Limited Investor Presentation Q4 & FY24
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

!pip install -q pymupdf

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import math
import platform
import re
import sys

import fitz
import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D11"

DOCUMENT_NAME = (
    "Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

SOURCE_FORMAT = ".pdf"

INPUT_REPRESENTATION = "Original investor-presentation PDF"

DIRECT_DOCUMENT_INGESTION = True

EXPECTED_PAGE_COUNT = 10


# ------------------------------------------------------------
# Fixed Stage 1 expectations
# ------------------------------------------------------------

EXPECTED_RECORD_COUNT = 199

EXPECTED_CATEGORY_COUNTS = {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 45,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7
}


# ------------------------------------------------------------
# Fixed Stage 1 schema
# ------------------------------------------------------------

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]


STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]


VALUE_ALLOWED_TYPES = (
    str,
    int,
    float,
    type(None)
)


MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]


ALLOWED_CATEGORIES = set(
    EXPECTED_CATEGORY_COUNTS
)


# ------------------------------------------------------------
# Fixed Stage 1 topic families
# ------------------------------------------------------------

EXPECTED_PRESENTATION_METADATA_TOPICS = [
    "Presentation title",
    "Company name",
    "Safe Harbor"
]


EXPECTED_MANAGEMENT_TOPICS = [
    "Market conditions",
    "Revenue from Operations",
    "Comparative Revenue from Operations",
    "Revenue mix — Fabric",
    "Revenue mix — Garments",
    "Revenue mix — Yarn & Others",
    "EBITDA",
    "EBITDA Margin",
    "Profit After Tax",
    "PAT Margin",
    "Retail footprint",
    "Sales promotion spending",
    "Comparative sales promotion spending",
    "Dividend",
    "Dividend percentage",
    "Face value",
    "Management spokesperson"
]


EXPECTED_QUARTERLY_TOPICS = {
    "Net Revenue",
    "EBITDA",
    "Net Profit After Tax"
}


EXPECTED_COMPANY_PROFILE_TOPICS = [
    "Company history",
    "Market position",
    "Product categories",
    "Brand portfolio",
    "Retail and online presence",
    "Manufacturing certifications",
    "Manufacturing locations",
    "Distribution ecosystem"
]


EXPECTED_TIMELINE_TOPICS = [
    "Established",
    "Public listing",
    "Tarapur capacity",
    "Siyaram brand promotion",
    "Oxemberg",
    "J. Hampstead",
    "Silvassa weaving capacity",
    "Mistair",
    "Most trusted brand",
    "Cadini",
    "Amravati unit",
    "Siyaram’s Mozzo",
    "Guinness World Record",
    "DEN-KNIT",
    "Tessio",
    "EVITA & BREEZY",
    "Ethnair"
]


EXPECTED_OPERATIONAL_TOPICS = [
    "Distributors",
    "Fabric sold",
    "Stores across nation",
    "Retail space",
    "Apparels sold",
    "Customers served",
    "End markets"
]


# ------------------------------------------------------------
# Reference values for targeted POST-extraction diagnostics
# ------------------------------------------------------------

EXPECTED_OPERATIONAL_VALUES = {
    "Distributors": "800+",
    "Fabric sold": "~100",
    "Stores across nation": "245+",
    "Retail space": "~1.85",
    "Apparels sold": "~4.5",
    "Customers served": "5 and counting"
}


EXPECTED_MANAGEMENT_RETAIL_FOOTPRINT = 247


EXPECTED_PAGE5_RECORDS_PER_METRIC = 15

EXPECTED_PAGE6_RECORD_COUNT = 102


# ------------------------------------------------------------
# Source markers used only for input-integrity diagnostics
# ------------------------------------------------------------

SOURCE_MARKER_PATTERNS = {
    "presentation_title":
        r"Investor Presentation",

    "safe_harbor":
        r"Safe Harbor",

    "management_commentary":
        r"Management Commentary",

    "quarterly_business_performance":
        r"Quarterly Business Performance",

    "profit_and_loss":
        r"Profit\s*&\s*Loss Statement",

    "company_profile":
        r"Our Legacy,\s*Our Future",

    "corporate_timeline":
        r"We Improve\.\s*Grow\.\s*Accelerate",

    "operational_footprint":
        r"We serve multiple end markets"
}


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs_D11_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_input_integrity.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_representation.json"
)

PAGE_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_page_diagnostics.csv"
)

PROMPT_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_prompt.txt"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_parsed_extraction.json"
)

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_technical_diagnostics.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_experiment_summary.json"
)


print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Representation:", INPUT_REPRESENTATION)
print("Expected pages:", EXPECTED_PAGE_COUNT)
print("Expected reference records:", EXPECTED_RECORD_COUNT)
print("Expected fields:", len(EXPECTED_FIELDS))

In [ ]:
# ============================================================
# 2. Source document and integrity diagnostics
# ============================================================

print(
    "Upload the original D11 investor-presentation PDF."
)


uploaded = files.upload()


pdf_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".pdf")
]


if len(pdf_paths) != 1:

    raise ValueError(
        "Upload exactly one PDF source document."
    )


SOURCE_PATH = pdf_paths[0]


# ------------------------------------------------------------
# SHA-256
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)


FILE_SIZE_BYTES = (
    SOURCE_PATH.stat().st_size
)


FILE_NON_EMPTY = (
    FILE_SIZE_BYTES > 0
)


# ------------------------------------------------------------
# PDF inspection
# ------------------------------------------------------------

pdf_document = fitz.open(
    SOURCE_PATH
)


PAGE_COUNT = len(
    pdf_document
)


PAGE_COUNT_VALID = (
    PAGE_COUNT
    == EXPECTED_PAGE_COUNT
)


page_rows = []

page_texts = []


for page_number, page in enumerate(
    pdf_document,
    start=1
):

    text = (
        page.get_text(
            "text"
        )
        or ""
    )


    images = page.get_images(
        full=True
    )


    drawings = page.get_drawings()


    page_texts.append(
        text
    )


    page_rows.append(
        {
            "Page Number":
                page_number,

            "Native Character Count":
                len(text),

            "Native Word Count":
                len(
                    text.split()
                ),

            "Image Count":
                len(images),

            "Drawing Count":
                len(drawings),

            "Width":
                float(
                    page.rect.width
                ),

            "Height":
                float(
                    page.rect.height
                ),

            "Rotation":
                int(
                    page.rotation
                )
        }
    )


page_diagnostics_df = pd.DataFrame(
    page_rows
)


FULL_TEXT = "\n".join(
    page_texts
)


TOTAL_NATIVE_CHARACTERS = len(
    FULL_TEXT
)


TOTAL_NATIVE_WORDS = len(
    FULL_TEXT.split()
)


TEXT_EXTRACTABLE = bool(
    TOTAL_NATIVE_CHARACTERS > 100
)


OCR_REQUIRED = (
    not TEXT_EXTRACTABLE
)


# ------------------------------------------------------------
# Source-marker diagnostics
# ------------------------------------------------------------

SOURCE_MARKER_STATUS = {
    name: bool(
        re.search(
            pattern,
            FULL_TEXT,
            flags=re.IGNORECASE
        )
    )

    for name, pattern
    in SOURCE_MARKER_PATTERNS.items()
}


ALL_SOURCE_MARKERS_PRESENT = all(
    SOURCE_MARKER_STATUS.values()
)


# ------------------------------------------------------------
# Page-role metadata
# ------------------------------------------------------------

PAGE_ROLES = {
    "1": "Presentation cover",
    "2": "Safe Harbor",
    "3": "Section divider — Q4 & FY24 Performance",
    "4": "Management Commentary",
    "5": "Quarterly Business Performance chart",
    "6": "Q4FY24 Profit & Loss Statement",
    "7": "Section divider — Our Legacy, Our Future",
    "8": "Company profile",
    "9": "Corporate timeline",
    "10": "Operational-footprint infographic"
}


EXCLUDED_DIVIDER_PAGES = [
    3,
    7
]


DIRECT_PDF_INGESTION_USABLE = all([
    FILE_NON_EMPTY,
    PAGE_COUNT_VALID,
    TEXT_EXTRACTABLE,
    ALL_SOURCE_MARKERS_PRESENT
])


INPUT_INTEGRITY_PASSED = (
    DIRECT_PDF_INGESTION_USABLE
)


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        SOURCE_PATH.name,

    "input_file_sha256":
        SOURCE_SHA256,

    "input_representation":
        INPUT_REPRESENTATION,

    "source_format":
        SOURCE_FORMAT,

    "file_size_bytes":
        FILE_SIZE_BYTES,

    "file_non_empty":
        FILE_NON_EMPTY,

    "expected_page_count":
        EXPECTED_PAGE_COUNT,

    "observed_page_count":
        PAGE_COUNT,

    "page_count_valid":
        PAGE_COUNT_VALID,

    "native_text_characters":
        TOTAL_NATIVE_CHARACTERS,

    "native_word_count":
        TOTAL_NATIVE_WORDS,

    "text_extractable":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "source_marker_status":
        SOURCE_MARKER_STATUS,

    "all_source_markers_present":
        ALL_SOURCE_MARKERS_PRESENT,

    "page_roles":
        PAGE_ROLES,

    "excluded_divider_pages":
        EXCLUDED_DIVIDER_PAGES,

    "contains_visual_financial_chart":
        True,

    "contains_financial_matrix_table":
        True,

    "contains_company_profile_layout":
        True,

    "contains_corporate_timeline":
        True,

    "contains_operational_infographic":
        True,

    "contains_repeated_financial_metrics":
        True,

    "contains_qualified_operational_values":
        True,

    "direct_pdf_ingestion_usable":
        DIRECT_PDF_INGESTION_USABLE,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED
}


INPUT_INTEGRITY_PATH.write_text(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


page_diagnostics_df.to_csv(
    PAGE_DIAGNOSTICS_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Source:",
    SOURCE_PATH.name
)

print(
    "SHA-256:",
    SOURCE_SHA256
)

print(
    "Observed pages:",
    PAGE_COUNT
)

print(
    "Page count valid:",
    PAGE_COUNT_VALID
)

print(
    "Native characters:",
    TOTAL_NATIVE_CHARACTERS
)

print(
    "Text extractable:",
    TEXT_EXTRACTABLE
)

print(
    "OCR required:",
    OCR_REQUIRED
)

print(
    "All source markers present:",
    ALL_SOURCE_MARKERS_PRESENT
)

print(
    "Input integrity passed:",
    INPUT_INTEGRITY_PASSED
)


display(
    page_diagnostics_df
)

if not FILE_NON_EMPTY:

    raise AssertionError(
        "D11 source PDF is empty."
    )


if not PAGE_COUNT_VALID:

    raise AssertionError(
        f"Expected {EXPECTED_PAGE_COUNT} pages, "
        f"found {PAGE_COUNT}."
    )


if not TEXT_EXTRACTABLE:

    raise AssertionError(
        "Expected D11 to contain a usable native text layer."
    )


if not ALL_SOURCE_MARKERS_PRESENT:

    raise AssertionError(
        "One or more required D11 source-region markers "
        "are missing."
    )

In [ ]:
# ============================================================
# 3. Branch A representation
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        SOURCE_PATH.name,

    "input_format":
        SOURCE_FORMAT,

    "source_representation":
        (
            "Investor-presentation PDF with native text, "
            "financial charts, financial tables, timeline "
            "and operational infographic"
        ),

    "complete_original_document_supplied":
        True,

    "direct_document_ingestion":
        True,

    "diagnostic_native_text_inspection_applied":
        True,

    "native_text_used_as_model_input":
        False,

    "diagnostic_page_layout_inspection_applied":
        True,

    "derived_representation_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "page_extraction_applied":
        False,

    "page_cropping_applied":
        False,

    "chart_reconstruction_applied":
        False,

    "table_reconstruction_applied":
        False,

    "timeline_reconstruction_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "model_input_description": (
        "The complete original ten-page D11 investor-presentation "
        "PDF is submitted directly to the LLM. Native-text and "
        "page-layout inspection are used only for notebook "
        "diagnostics. No extracted text, reconstructed chart, "
        "reconstructed table or other derived representation is "
        "supplied to the model."
    )
}


REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 4. Extraction prompt
# ============================================================

BRANCH_A_PROMPT = """You are an information extraction assistant.

Extract the predefined financial, corporate-profile, historical and
operational records represented within the defined scope of the
attached original PDF investor presentation:

Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24.

Treat the attached original PDF as the only source of information.

For every included record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location

Use exactly one of these Category values:

- Presentation metadata
- Management commentary
- Quarterly business performance
- Profit and loss statement
- Company profile
- Corporate timeline
- Operational footprint


1. Presentation metadata

From physical PDF pages 1 and 2, extract the predefined metadata
concepts concerning:

- the presentation title;
- the company name;
- the Safe Harbor status/purpose statement.

Represent the Safe Harbor material as one principal metadata record.
Do not extract the complete legal disclaimer sentence-by-sentence.


2. Management commentary

From physical PDF page 4, extract the explicitly represented
management-commentary observations concerning:

- market conditions;
- Revenue from Operations and its comparative period;
- the revenue mix by Fabric, Garments, and Yarn & Others;
- EBITDA and EBITDA Margin;
- Profit After Tax and PAT Margin;
- retail footprint;
- current and comparative sales-promotion spending;
- the approved dividend;
- the dividend percentage;
- the share face value associated with the dividend statement;
- the identified management spokesperson.

Extract the values and periods directly from the page.

Keep independently represented observations separate even when a
similar metric occurs elsewhere in the presentation.

Do not use values from another page to complete this source section.


3. Quarterly Business Performance

From the chart on physical PDF page 5, extract every explicitly printed
annual total and every explicitly printed quarterly component for:

- Net Revenue;
- EBITDA;
- Net Profit After Tax.

For each metric:

- preserve the annual totals for each represented fiscal year;
- preserve every explicitly labelled Q1, Q2, Q3 and Q4 component;
- associate each quarterly value with the correct fiscal year;
- preserve the represented monetary scale;
- preserve the chart's stated qualification concerning standalone
  financials and rounding where relevant in Description.

Extract only values numerically printed in the chart.

Do not estimate values from bar height, bar area, graphical position,
colour, or proportional size.

Annual totals and quarterly components are separate records.

Do not merge chart observations with similar values represented in
management commentary or the Profit & Loss Statement.


4. Q4FY24 Profit & Loss Statement

From the table on physical PDF page 6, extract every explicitly
populated numerical observation represented in the body of the table.

Use the financial row label as Topic.

For the main period columns, preserve observations under:

- Q4 FY24;
- Q4 FY23;
- Q3 FY24;
- FY24;
- FY23.

For populated YoY and QoQ cells:

- create separate records;
- distinguish Year-on-Year from Quarter-on-Quarter change in Topic;
- preserve the correct comparison in Reporting Period;
- preserve percentages as percentages.

Do not create records for blank YoY or QoQ cells.

Treat margin rows as percentages.

Treat EPS using its represented rupee-per-share scale rather than
the table-level Rs. Mn scale.

Also extract the two explicitly represented marketing and sales
promotion expense observations in the page-6 footnote.

Do not calculate any missing YoY, QoQ, margin or other value.

Do not recompute or reconcile totals.

Do not merge page-6 observations with similar values printed on
other pages.


5. Company profile

From physical PDF page 8, extract the principal represented
company-profile observations concerning:

- company history;
- market position;
- product categories;
- explicitly listed brands and sub-brands;
- retail and online presence;
- manufacturing certifications;
- manufacturing locations;
- distribution ecosystem.

Preserve explicitly represented names, locations and certification
wording.

Do not create records from decorative imagery or the closing tagline.


6. Corporate timeline

From physical PDF page 9, extract every explicitly listed milestone
within the four represented timeline phases.

Preserve:

- the milestone subject as Topic;
- the represented milestone wording in Description or Value;
- the phase period as Reporting Period.

Do not infer exact event years where only a phase-level period is
represented.

Do not create records from decorative images or phase numbering alone.


7. Operational footprint

From physical PDF page 10, extract every prominently represented
operational metric concerning:

- distributors;
- fabric sold;
- stores across the nation;
- retail space;
- apparels sold;
- customers served.

Also extract one record representing the explicitly listed commercial
channels/end markets.

Preserve approximation, lower-bound and continuation wording exactly
when it forms part of a represented value.

For example, if a printed value contains an approximation symbol,
a plus sign or wording such as “and counting”, preserve that
qualification rather than silently converting it into an exact number.

Do not replace a page-10 observation with a similar value represented
elsewhere in the document.


Excluded source regions

Physical PDF pages 3 and 7 are section-divider slides and do not
contribute extraction records within this task.

Decorative imagery and logos are outside the extraction scope.


Field rules:

Category:
- Use exactly one of the seven Category labels defined above.

Topic:
- Use a concise stable label describing the represented metric,
  statement, milestone or concept.
- Preserve source terminology for financial metrics.

Description:
- Provide a concise source-grounded description of the observation.
- Preserve material source qualifications where relevant.
- Do not introduce external interpretation.

Value:
- Use a JSON number when the source represents an unqualified numeric
  value.
- Use a JSON string when the value is textual or when qualification
  such as "~", "+", or "and counting" is semantically part of the
  represented value.
- Use null only where no separate Value applies.
- Preserve negative signs.
- Do not calculate, estimate, derive, convert or correct values.

Unit:
- Preserve the represented measurement scale.
- Use consistent source-grounded forms such as:
  text
  Rs. Mn
  Rs. crores
  percent
  stores
  distributors
  Mn meters
  L sqft
  Mn pieces
  Mn customers
  Rs. per share
  Rs.
  year
- Do not silently rescale monetary values.

Reporting Period:
- Derive the period directly from the represented source.
- Preserve fiscal-year and quarter distinctions.
- Preserve comparison periods for YoY and QoQ records.
- Use the represented timeline phase period for timeline milestones.
- Do not infer an exact year where only a phase period is represented.

Source Location:
- Use the physical PDF page containing the observation.
- Use the form:
  "PDF page N"

Additional rules:

- Use only information explicitly represented in the supplied PDF.
- Preserve repeated observations when they occur independently in
  different source sections.
- Do not deduplicate records solely because Topic or Value is repeated.
- Do not estimate chart values visually.
- Do not extract blank table cells.
- Preserve negative percentages.
- Preserve approximation and lower-bound wording.
- Do not use external knowledge.
- Do not follow external links.
- Do not calculate missing values.
- Do not normalise or convert measurement scales.
- Do not silently correct source wording.
- Do not extract records outside the predefined source scope.
- Verify that every item within the defined scope has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D11",
  "branch": "A",
  "records": [
    {
      "Category": null,
      "Topic": null,
      "Description": null,
      "Value": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_A_PROMPT,
    encoding="utf-8"
)


PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)


print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

print()

print(
    BRANCH_A_PROMPT
)

## Independent Branch A extraction

Open a new independent ChatGPT conversation.

Upload:

1. the complete original D11 PDF;
2. `D11_branch_A_prompt.txt`.

Submit the prompt once.

Save complete untouched model response as:

`D11_branch_A_raw_response.txt`


In [ ]:
# ============================================================
# 5. Raw response preservation and parsing
# ============================================================

print(
    "Upload the untouched "
    "D11_branch_A_raw_response.txt file."
)


uploaded = files.upload()


txt_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".txt")
]


if len(txt_paths) != 1:

    raise ValueError(
        "Upload exactly one TXT raw-response file."
    )


UPLOADED_RAW_RESPONSE_PATH = (
    txt_paths[0]
)


raw_response_text = (
    UPLOADED_RAW_RESPONSE_PATH.read_text(
        encoding="utf-8"
    )
)


if not raw_response_text.strip():

    raise ValueError(
        "The uploaded raw response is empty."
    )


# ------------------------------------------------------------
# Preserve untouched response BEFORE parsing
# ------------------------------------------------------------

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


# ------------------------------------------------------------
# Non-crashing JSON parse
# ------------------------------------------------------------

valid_json = False

json_parsing_error = None

parsed_response = None


try:

    parsed_response = json.loads(
        raw_response_text
    )

    valid_json = True


except json.JSONDecodeError as error:

    json_parsing_error = str(
        error
    )


# ------------------------------------------------------------
# Standard wrapper
# ------------------------------------------------------------

top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)


document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)


document_id_correct = (
    top_level_object_valid
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)


branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)


branch_correct = (
    top_level_object_valid
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)


records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)


records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)


records_evaluable = (
    valid_json
    and top_level_object_valid
    and records_present
    and records_is_list
)


if records_evaluable:

    extracted_records = (
        parsed_response[
            "records"
        ]
    )

    observed_record_count = len(
        extracted_records
    )


else:

    extracted_records = []

    observed_record_count = None


# ------------------------------------------------------------
# Parsed artifact only when records are evaluable
# ------------------------------------------------------------

parsed_extraction_created = False

parsed_extraction_sha256 = None


if records_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get(
                "document_id"
            ),

        "branch":
            parsed_response.get(
                "branch"
            ),

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_created = True

    parsed_extraction_sha256 = (
        sha256_file(
            PARSED_EXTRACTION_PATH
        )
    )


print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)

print(
    "Valid JSON:",
    valid_json
)

print(
    "JSON parsing error:",
    json_parsing_error
)

print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed record count:",
    observed_record_count
)


if records_evaluable:

    extracted_df = pd.DataFrame(
        extracted_records
    )

    display(
        extracted_df
    )

In [ ]:
# ============================================================
# 6. Record and content diagnostics
# ============================================================

record_structure_issues = []

field_type_issues = []

missing_mandatory_values = []


# ------------------------------------------------------------
# A. Exact schema
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Record is not a JSON object"
                }
            )

            continue


        observed_fields = list(
            record.keys()
        )


        if observed_fields != EXPECTED_FIELDS:

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Field names or field order differ",

                    "expected_fields":
                        EXPECTED_FIELDS,

                    "observed_fields":
                        observed_fields,

                    "missing_fields":
                        [
                            field
                            for field
                            in EXPECTED_FIELDS
                            if field not in record
                        ],

                    "extra_fields":
                        [
                            field
                            for field
                            in observed_fields
                            if field not in EXPECTED_FIELDS
                        ]
                }
            )


    records_with_structure_issues = len({
        issue["record_index"]
        for issue
        in record_structure_issues
    })


    record_schema_valid = (
        records_with_structure_issues == 0
    )


else:

    records_with_structure_issues = None

    record_schema_valid = None


# ------------------------------------------------------------
# B. Field types and mandatory content
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            continue


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )


            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(value).__name__,

                        "expected_type":
                            "string or null"
                    }
                )


        value = record.get(
            "Value"
        )


        if (
            isinstance(value, bool)
            or not isinstance(
                value,
                VALUE_ALLOWED_TYPES
            )
        ):

            field_type_issues.append(
                {
                    "record_index":
                        record_index,

                    "field":
                        "Value",

                    "observed_type":
                        type(value).__name__,

                    "expected_type":
                        "string, number or null"
                }
            )


        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(
                field
            )


            if (
                value is None
                or (
                    isinstance(
                        value,
                        str
                    )
                    and not value.strip()
                )
            ):

                missing_mandatory_values.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field
                    }
                )


    records_with_type_issues = len({
        issue["record_index"]
        for issue
        in field_type_issues
    })


    field_types_valid = (
        records_with_type_issues == 0
    )


    missing_mandatory_value_count = len(
        missing_mandatory_values
    )


    mandatory_fields_complete = (
        missing_mandatory_value_count == 0
    )


else:

    records_with_type_issues = None

    field_types_valid = None

    missing_mandatory_value_count = None

    mandatory_fields_complete = None


# ------------------------------------------------------------
# C. Counts and categories
# ------------------------------------------------------------

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


else:

    record_count_valid = None

    observed_category_counts = None

    categories_valid = None

    category_counts_valid = None


# ------------------------------------------------------------
# D. Exact full-record duplicates
# ------------------------------------------------------------

if records_evaluable:

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field
            in EXPECTED_FIELDS
        )
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    )


    duplicate_records = [
        list(key)
        for key, count
        in duplicate_counter.items()
        if count > 1
    ]


    duplicate_record_count = len(
        duplicate_records
    )


    duplicate_records_absent = (
        duplicate_record_count == 0
    )


else:

    duplicate_records = None

    duplicate_record_count = None

    duplicate_records_absent = None


# ------------------------------------------------------------
# E. Value-type counts
# ------------------------------------------------------------

if records_evaluable:

    numeric_value_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(record, dict)
            and isinstance(
                record.get("Value"),
                (int, float)
            )
            and not isinstance(
                record.get("Value"),
                bool
            )
        )
    )


    text_value_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(record, dict)
            and isinstance(
                record.get("Value"),
                str
            )
        )
    )


    null_value_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Value") is None
        )
    )


    negative_numeric_value_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(record, dict)
            and isinstance(
                record.get("Value"),
                (int, float)
            )
            and not isinstance(
                record.get("Value"),
                bool
            )
            and record.get("Value") < 0
        )
    )


else:

    numeric_value_count = None
    text_value_count = None
    null_value_count = None
    negative_numeric_value_count = None


# ------------------------------------------------------------
# F. Helpers
# ------------------------------------------------------------

def matching_records(
    category=None,
    topic=None,
    reporting_period=None,
    source_location=None
):

    if not records_evaluable:
        return []


    matches = []


    for record in extracted_records:

        if not isinstance(
            record,
            dict
        ):
            continue


        if (
            category is not None
            and record.get("Category")
            != category
        ):
            continue


        if (
            topic is not None
            and record.get("Topic")
            != topic
        ):
            continue


        if (
            reporting_period is not None
            and record.get("Reporting Period")
            != reporting_period
        ):
            continue


        if (
            source_location is not None
            and record.get("Source Location")
            != source_location
        ):
            continue


        matches.append(record)


    return matches


def unique_topic_record(
    category,
    topic
):

    matches = matching_records(
        category=category,
        topic=topic
    )


    if len(matches) == 1:
        return matches[0]


    return None


# ------------------------------------------------------------
# G. Metadata coverage
# ------------------------------------------------------------

if records_evaluable:

    presentation_metadata_status = {
        topic: (
            len(
                matching_records(
                    category="Presentation metadata",
                    topic=topic
                )
            )
            == 1
        )
        for topic
        in EXPECTED_PRESENTATION_METADATA_TOPICS
    }


    presentation_metadata_complete = all(
        presentation_metadata_status.values()
    )


else:

    presentation_metadata_status = None
    presentation_metadata_complete = None


# ------------------------------------------------------------
# H. Management-commentary topic coverage
# ------------------------------------------------------------

if records_evaluable:

    management_topic_status = {
        topic: (
            len(
                matching_records(
                    category="Management commentary",
                    topic=topic
                )
            )
            == 1
        )
        for topic
        in EXPECTED_MANAGEMENT_TOPICS
    }


    management_topics_complete = all(
        management_topic_status.values()
    )


else:

    management_topic_status = None
    management_topics_complete = None


# ------------------------------------------------------------
# I. Page-5 chart diagnostics
# ------------------------------------------------------------

if records_evaluable:

    quarterly_metric_counts = {
        topic: len(
            matching_records(
                category="Quarterly business performance",
                topic=topic
            )
        )
        for topic
        in EXPECTED_QUARTERLY_TOPICS
    }


    quarterly_metric_counts_valid = all(
        count
        == EXPECTED_PAGE5_RECORDS_PER_METRIC
        for count
        in quarterly_metric_counts.values()
    )


    quarterly_scope_complete = (
        sum(
            quarterly_metric_counts.values()
        )
        == EXPECTED_CATEGORY_COUNTS[
            "Quarterly business performance"
        ]
    )


else:

    quarterly_metric_counts = None
    quarterly_metric_counts_valid = None
    quarterly_scope_complete = None


# ------------------------------------------------------------
# J. Page-6 P&L diagnostics
# ------------------------------------------------------------

if records_evaluable:

    pnl_records = [
        record
        for record
        in extracted_records
        if (
            isinstance(record, dict)
            and record.get("Category")
            == "Profit and loss statement"
        )
    ]


    observed_pnl_record_count = len(
        pnl_records
    )


    blank_table_cells_not_extracted = (
        observed_pnl_record_count
        == EXPECTED_PAGE6_RECORD_COUNT
    )


    pnl_negative_changes_present = any(
        isinstance(
            record.get("Value"),
            (int, float)
        )
        and not isinstance(
            record.get("Value"),
            bool
        )
        and record.get("Value") < 0
        for record
        in pnl_records
    )


else:

    observed_pnl_record_count = None
    blank_table_cells_not_extracted = None
    pnl_negative_changes_present = None


# ------------------------------------------------------------
# K. Company-profile coverage
# ------------------------------------------------------------

if records_evaluable:

    company_profile_status = {
        topic: (
            len(
                matching_records(
                    category="Company profile",
                    topic=topic
                )
            )
            == 1
        )
        for topic
        in EXPECTED_COMPANY_PROFILE_TOPICS
    }


    company_profile_complete = all(
        company_profile_status.values()
    )


else:

    company_profile_status = None
    company_profile_complete = None


# ------------------------------------------------------------
# L. Timeline coverage
# ------------------------------------------------------------

if records_evaluable:

    timeline_topic_status = {
        topic: (
            len(
                matching_records(
                    category="Corporate timeline",
                    topic=topic
                )
            )
            == 1
        )
        for topic
        in EXPECTED_TIMELINE_TOPICS
    }


    timeline_complete = all(
        timeline_topic_status.values()
    )


else:

    timeline_topic_status = None
    timeline_complete = None


# ------------------------------------------------------------
# M. Operational-footprint coverage
# ------------------------------------------------------------

if records_evaluable:

    operational_topic_status = {
        topic: (
            len(
                matching_records(
                    category="Operational footprint",
                    topic=topic
                )
            )
            == 1
        )
        for topic
        in EXPECTED_OPERATIONAL_TOPICS
    }


    operational_scope_complete = all(
        operational_topic_status.values()
    )


else:

    operational_topic_status = None
    operational_scope_complete = None


# ------------------------------------------------------------
# N. Qualified-value preservation
# ------------------------------------------------------------

if records_evaluable:

    operational_value_status = {}


    for topic, expected_value in (
        EXPECTED_OPERATIONAL_VALUES.items()
    ):

        record = unique_topic_record(
            "Operational footprint",
            topic
        )


        operational_value_status[
            topic
        ] = (
            record is not None
            and record.get("Value")
            == expected_value
        )


    qualified_values_preserved = all(
        operational_value_status.values()
    )


else:

    operational_value_status = None
    qualified_values_preserved = None


# ------------------------------------------------------------
# O. Distinct store-scope diagnostic
# ------------------------------------------------------------

if records_evaluable:

    management_store_record = (
        unique_topic_record(
            "Management commentary",
            "Retail footprint"
        )
    )


    operational_store_record = (
        unique_topic_record(
            "Operational footprint",
            "Stores across nation"
        )
    )


    management_store_value_preserved = (
        management_store_record
        is not None
        and management_store_record.get(
            "Value"
        )
        == EXPECTED_MANAGEMENT_RETAIL_FOOTPRINT
    )


    operational_store_value_preserved = (
        operational_store_record
        is not None
        and operational_store_record.get(
            "Value"
        )
        == EXPECTED_OPERATIONAL_VALUES[
            "Stores across nation"
        ]
    )


    distinct_store_observations_preserved = all([
        management_store_value_preserved,
        operational_store_value_preserved
    ])


else:

    management_store_value_preserved = None
    operational_store_value_preserved = None
    distinct_store_observations_preserved = None


# ------------------------------------------------------------
# P. Divider pages excluded
# ------------------------------------------------------------

if records_evaluable:

    divider_page_issues = []


    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            continue


        location = str(
            record.get(
                "Source Location",
                ""
            )
        )


        if re.search(
            r"\bPDF page (3|7)\b",
            location,
            flags=re.IGNORECASE
        ):

            divider_page_issues.append(
                {
                    "record_index":
                        record_index,

                    "source_location":
                        location
                }
            )


    divider_page_issue_count = len(
        divider_page_issues
    )


    divider_pages_excluded = (
        divider_page_issue_count == 0
    )


else:

    divider_page_issues = None
    divider_page_issue_count = None
    divider_pages_excluded = None


# ------------------------------------------------------------
# Q. Repeated-metric preservation across source sections
# ------------------------------------------------------------

if records_evaluable:

    revenue_management_present = (
        len(
            matching_records(
                category="Management commentary",
                topic="Revenue from Operations"
            )
        )
        >= 1
    )


    revenue_pnl_present = (
        len(
            matching_records(
                category="Profit and loss statement",
                topic="Revenue from Operations"
            )
        )
        >= 1
    )


    ebitda_management_present = (
        len(
            matching_records(
                category="Management commentary",
                topic="EBITDA"
            )
        )
        >= 1
    )


    ebitda_chart_present = (
        len(
            matching_records(
                category="Quarterly business performance",
                topic="EBITDA"
            )
        )
        >= 1
    )


    ebitda_pnl_present = (
        len(
            matching_records(
                category="Profit and loss statement",
                topic="EBITDA"
            )
        )
        >= 1
    )


    repeated_financial_metrics_preserved = all([
        revenue_management_present,
        revenue_pnl_present,
        ebitda_management_present,
        ebitda_chart_present,
        ebitda_pnl_present
    ])


else:

    repeated_financial_metrics_preserved = None


# ------------------------------------------------------------
# R. Content diagnostics
# ------------------------------------------------------------

CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference":
        record_count_valid,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records_absent":
        duplicate_records_absent,

    "numeric_value_count":
        numeric_value_count,

    "text_value_count":
        text_value_count,

    "null_value_count":
        null_value_count,

    "negative_numeric_value_count":
        negative_numeric_value_count,

    "presentation_metadata_status":
        presentation_metadata_status,

    "presentation_metadata_complete":
        presentation_metadata_complete,

    "management_topic_status":
        management_topic_status,

    "management_topics_complete":
        management_topics_complete,

    "quarterly_metric_counts":
        quarterly_metric_counts,

    "quarterly_metric_counts_valid":
        quarterly_metric_counts_valid,

    "quarterly_scope_complete":
        quarterly_scope_complete,

    "observed_pnl_record_count":
        observed_pnl_record_count,

    "blank_table_cells_not_extracted":
        blank_table_cells_not_extracted,

    "pnl_negative_changes_present":
        pnl_negative_changes_present,

    "company_profile_status":
        company_profile_status,

    "company_profile_complete":
        company_profile_complete,

    "timeline_topic_status":
        timeline_topic_status,

    "timeline_complete":
        timeline_complete,

    "operational_topic_status":
        operational_topic_status,

    "operational_scope_complete":
        operational_scope_complete,

    "operational_value_status":
        operational_value_status,

    "qualified_values_preserved":
        qualified_values_preserved,

    "management_store_value_preserved":
        management_store_value_preserved,

    "operational_store_value_preserved":
        operational_store_value_preserved,

    "distinct_store_observations_preserved":
        distinct_store_observations_preserved,

    "divider_page_issue_count":
        divider_page_issue_count,

    "divider_pages_excluded":
        divider_pages_excluded,

    "repeated_financial_metrics_preserved":
        repeated_financial_metrics_preserved
}


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Observed records:",
    observed_record_count
)

print(
    "Record count matches:",
    record_count_valid
)

print(
    "Category counts match:",
    category_counts_valid
)

print(
    "Missing mandatory values:",
    missing_mandatory_value_count
)

print(
    "Duplicate records:",
    duplicate_record_count
)

print(
    "Metadata complete:",
    presentation_metadata_complete
)

print(
    "Management commentary complete:",
    management_topics_complete
)

print(
    "Quarterly chart scope complete:",
    quarterly_scope_complete
)

print(
    "Page-6 blank cells excluded:",
    blank_table_cells_not_extracted
)

print(
    "Company profile complete:",
    company_profile_complete
)

print(
    "Timeline complete:",
    timeline_complete
)

print(
    "Operational scope complete:",
    operational_scope_complete
)

print(
    "Qualified operational values preserved:",
    qualified_values_preserved
)

print(
    "Distinct store observations preserved:",
    distinct_store_observations_preserved
)

print(
    "Divider pages excluded:",
    divider_pages_excluded
)

print(
    "Repeated financial metrics preserved:",
    repeated_financial_metrics_preserved
)


print(
    "\nObserved category counts:"
)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
    if observed_category_counts
    is not None
    else None
)

In [ ]:
# ============================================================
# 7. Technical diagnostic summary and experiment metadata
# ============================================================

# ------------------------------------------------------------
# Technical/schema validity ONLY
# ------------------------------------------------------------

STRUCTURAL_CHECKS = {
    "valid_json":
        bool(valid_json),

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_present":
        bool(document_id_present),

    "document_id_correct":
        bool(document_id_correct),

    "branch_present":
        bool(branch_present),

    "branch_correct":
        bool(branch_correct),

    "records_present":
        bool(records_present),

    "records_is_list":
        bool(records_is_list),

    "record_schema_valid":
        (
            record_schema_valid
            if records_evaluable
            else None
        ),

    "field_types_valid":
        (
            field_types_valid
            if records_evaluable
            else None
        )
}


structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])


# ------------------------------------------------------------
# Structure check
# ------------------------------------------------------------

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "records_evaluable":
        records_evaluable,

    "structural_checks":
        STRUCTURAL_CHECKS,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_valid":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issue_count":
        (
            len(field_type_issues)
            if records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "missing_mandatory_values":
        (
            missing_mandatory_values
            if records_evaluable
            else None
        ),

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records":
        duplicate_records,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structurally_evaluable":
        bool(structurally_evaluable)
}


TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment metadata
# ------------------------------------------------------------

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_page_count":
            EXPECTED_PAGE_COUNT,

        "observed_page_count":
            PAGE_COUNT,

        "page_count_verified":
            PAGE_COUNT_VALID,

        "text_extractable":
            TEXT_EXTRACTABLE,

        "ocr_required":
            OCR_REQUIRED,

        "contains_financial_chart":
            True,

        "contains_financial_table":
            True,

        "contains_company_profile":
            True,

        "contains_corporate_timeline":
            True,

        "contains_operational_infographic":
            True,

        "excluded_divider_pages":
            EXCLUDED_DIVIDER_PAGES
    },

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "diagnostic_native_text_inspection_applied":
        True,

    "native_text_used_as_model_input":
        False,

    "diagnostic_page_layout_inspection_applied":
        True,

    "derived_representation_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "pdf_to_text_conversion_applied":
        False,

    "page_extraction_applied":
        False,

    "page_cropping_applied":
        False,

    "chart_reconstruction_applied":
        False,

    "table_reconstruction_applied":
        False,

    "timeline_reconstruction_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "complete_original_pdf_supplied":
        True,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "reference_expectations_disclosed_to_model":
        False,

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED,

    "representation_file":
        REPRESENTATION_PATH.name,

    "page_diagnostics_file":
        PAGE_DIAGNOSTICS_PATH.name,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        (
            "JSON object with document_id, "
            "branch and records"
        ),

    "execution_environment":
        "Independent ChatGPT conversation",

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(structurally_evaluable),


    "notes": (
        "Branch A submits the complete original ten-page D11 "
        "investor-presentation PDF directly to the model. Native-text "
        "and page-layout inspection are used only for source-integrity "
        "and representation diagnostics and are not supplied as an "
        "alternative model representation. No OCR, PDF-to-text "
        "conversion, page extraction, cropping, chart reconstruction, "
        "table reconstruction, structural conversion, normalisation, "
        "unit conversion or manual correction is applied before "
        "extraction. Stage 1 reference values, expected record count, "
        "expected category distribution, reference periods and "
        "reference source locations are not supplied to the model. "
        "Content-level validation is performed separately in Validation A — D11."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment summary
# ------------------------------------------------------------

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        INPUT_INTEGRITY_PASSED,

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "text_extractable":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "structurally_evaluable":
        bool(structurally_evaluable),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "presentation_metadata_complete":
        presentation_metadata_complete,

    "management_topics_complete":
        management_topics_complete,

    "quarterly_scope_complete":
        quarterly_scope_complete,

    "quarterly_metric_counts_valid":
        quarterly_metric_counts_valid,

    "observed_pnl_record_count":
        observed_pnl_record_count,

    "blank_table_cells_not_extracted":
        blank_table_cells_not_extracted,

    "pnl_negative_changes_present":
        pnl_negative_changes_present,

    "company_profile_complete":
        company_profile_complete,

    "timeline_complete":
        timeline_complete,

    "operational_scope_complete":
        operational_scope_complete,

    "qualified_values_preserved":
        qualified_values_preserved,

    "distinct_store_observations_preserved":
        distinct_store_observations_preserved,

    "divider_pages_excluded":
        divider_pages_excluded,

    "repeated_financial_metrics_preserved":
        repeated_financial_metrics_preserved,

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        parsed_extraction_created,

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, representation "
        "characterisation, D11 Branch A direct-PDF execution "
        "preservation, technical/schema checks and document-specific "
        "content diagnostics only. Formal agreement with the fixed "
        "Stage 1 reference dataset is evaluated separately in "
        "Validation A — D11."
    )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print(
    "Structural checks:"
)

print(
    json.dumps(
        STRUCTURAL_CHECKS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nContent diagnostics:"
)

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\n" + "=" * 60
)

print(
    "D11 Branch A experiment completed"
)

print(
    "=" * 60
)


print(
    "Input integrity passed        :",
    INPUT_INTEGRITY_PASSED
)

print(
    "Text extractable              :",
    TEXT_EXTRACTABLE
)

print(
    "OCR required                  :",
    OCR_REQUIRED
)

print(
    "Raw response preserved        :",
    RAW_RESPONSE_PATH.exists()
)

print(
    "Valid JSON                    :",
    valid_json
)

print(
    "Records evaluable             :",
    records_evaluable
)

print(
    "Expected records              :",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records              :",
    (
        observed_record_count
        if observed_record_count
        is not None
        else "Not evaluable"
    )
)

print(
    "Record count matches          :",
    record_count_valid
)

print(
    "Category counts match         :",
    category_counts_valid
)

print(
    "Record schema valid           :",
    record_schema_valid
)

print(
    "Field types valid             :",
    field_types_valid
)

print(
    "Structurally evaluable       :",
    structurally_evaluable
)

print(
    "Content validation performed : False"
)

print(
    "Next step                     : Validation A — D11"
)


# ------------------------------------------------------------
# Output existence
# ------------------------------------------------------------

required_output_paths = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PAGE_DIAGNOSTICS_PATH,
    PROMPT_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if (
    parsed_extraction_created
    and PARSED_EXTRACTION_PATH.exists()
):

    required_output_paths.append(
        PARSED_EXTRACTION_PATH
    )


missing_output_files = [
    path.name
    for path
    in required_output_paths
    if not path.exists()
]


if missing_output_files:

    raise AssertionError(
        "Missing output files: "
        f"{missing_output_files}"
    )


print(
    "\nGenerated D11 Branch A files:\n"
)


for path in required_output_paths:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )